In [4]:
import mysql.connector

# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",  # 替换为你自己的密码
    database="ucb_db"
)

cursor = conn.cursor()

# 查询所有描述
cursor.execute("SELECT description FROM business_descriptions")
descriptions = [row[0] for row in cursor.fetchall() if row[0]]

# 筛选包含 Los Angeles 的描述
la_descriptions = [desc for desc in descriptions if "Los Angeles" in desc]

print(f"✅ 共找到 {len(la_descriptions)} 条包含 'Los Angeles' 的描述")
for desc in la_descriptions[:5]:
    print("📝", desc)


✅ 共找到 7 条包含 'Los Angeles' 的描述
📝  This textile exporter located in Los Angeles, CA 90023 specializes in providing high-quality fabrics and materials to global markets.
📝  This vibrant Korean restaurant in Los Angeles, CA 90005, offers an authentic taste of Korea with a diverse menu featuring traditional dishes and modern interpretations, perfect for both casual dining and special occasions.
📝  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles, perfect for all your sewing and crafting needs.
📝  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles and materials for all your sewing and crafting needs.
📝  Discover a vibrant bead store in Los Angeles, CA 90014, offering a wide selection of high-quality beads, supplies, and crafting tools for all your creative projects.


In [5]:
import mysql.connector
import openai
import time
import json
from tqdm import tqdm
from openai import AzureOpenAI

# ====== Azure OpenAI 配置 ======
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)

deployment_name = "gpt-4o-mini"  # 在 Azure Portal 的 Deployments 中查看


# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",
    database="ucb_db"
)
cursor = conn.cursor()

# 获取所有字段（包括 description）
cursor.execute("SELECT * FROM business_descriptions")
column_names = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()

# 定义 GPT 判断函数
def is_los_angeles(description, client, deployment_name):
    # 构造 prompt
    prompt = (
        "Does the following business description mention that it is located in Los Angeles?\n"
        "Just answer 'yes' or 'no'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts geographic location from business descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower().startswith("yes")

    except Exception as e:
        print(f"❌ Error determining location from description: {e}")
        return False

# ✅ 用于保存符合条件的记录
la_data = []

# ✅ 遍历记录进行 GPT 判断
for i, row in enumerate(rows[:100]):  # 可根据需要扩大范围
    record = dict(zip(column_names, row))
    description = record.get("description", "")

    if description and is_los_angeles(description, client, deployment_name):
        la_data.append(record)
        print(f"✅ [{i}] 是洛杉矶: {description}")
    else:
        print(f"❌ [{i}] 不是洛杉矶")

print(f"\n🎯 最终保留 {len(la_data)} 条位于洛杉矶的完整记录")


✅ [0] 是洛杉矶:  This textile exporter located in Los Angeles, CA 90023 specializes in providing high-quality fabrics and materials to global markets.
✅ [1] 是洛杉矶:  This vibrant Korean restaurant in Los Angeles, CA 90005, offers an authentic taste of Korea with a diverse menu featuring traditional dishes and modern interpretations, perfect for both casual dining and special occasions.
✅ [2] 是洛杉矶:  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles, perfect for all your sewing and crafting needs.
✅ [3] 是洛杉矶:  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles and materials for all your sewing and crafting needs.
❌ [4] 不是洛杉矶
❌ [5] 不是洛杉矶
❌ [6] 不是洛杉矶
❌ [7] 不是洛杉矶
❌ [8] 不是洛杉矶
❌ [9] 不是洛杉矶
❌ [10] 不是洛杉矶
❌ [11] 不是洛杉矶
❌ [12] 不是洛杉矶
❌ [13] 不是洛杉矶
❌ [14] 不是洛杉矶
❌ [15] 不是洛杉矶
❌ [16] 不是洛杉矶
❌ [17] 不是洛杉矶
❌ [18] 不是洛杉矶
❌ [19] 不是洛杉矶
❌ [20] 不是洛杉矶
❌ [21] 不是洛杉矶
❌ [22] 不是洛杉矶
❌ [23] 不是洛杉矶
❌ [24] 不是洛杉矶
❌ [25] 不是洛杉矶
❌ [26] 不是洛杉矶
❌ [27] 不是洛杉

In [6]:
la_data

[{'name': 'City Textile',
  'gmap_id': 'gmap_0',
  'description': ' This textile exporter located in Los Angeles, CA 90023 specializes in providing high-quality fabrics and materials to global markets.',
  'num_of_reviews': 6,
  'hours': 'null',
  'MISC': 'null',
  'state': 'Open now'},
 {'name': 'San Soo Dang',
  'gmap_id': 'gmap_1',
  'description': ' This vibrant Korean restaurant in Los Angeles, CA 90005, offers an authentic taste of Korea with a diverse menu featuring traditional dishes and modern interpretations, perfect for both casual dining and special occasions.',
  'num_of_reviews': 18,
  'hours': '[["Thursday", "6:30AM–6PM"], ["Friday", "6:30AM–6PM"], ["Saturday", "6:30AM–6PM"], ["Sunday", "7AM–12PM"], ["Monday", "Closed"], ["Tuesday", "6:30AM–6PM"], ["Wednesday", "6:30AM–6PM"]]',
  'MISC': '{"Amenities": ["Good for kids"], "Offerings": ["Comfort food"], "Atmosphere": ["Casual"], "Accessibility": ["Wheelchair accessible entrance"], "Service options": ["Takeout", "Dine-in", 

In [7]:
import sqlite3

# 尝试连接 SQLite 数据库
db_path = "../query_dataset/review_query.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 查看数据库中所有表
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("📋 数据库中包含的表：", tables)


📋 数据库中包含的表： [('review',)]


In [8]:
cursor.execute("SELECT * FROM review LIMIT 5;")
rows = cursor.fetchall()
colnames = [desc[0] for desc in cursor.description]

print("字段名：", colnames)
for row in rows:
    print(row)


字段名： ['name', 'time', 'rating', 'text', 'gmap_id']
('Michael Rizal', 'September 03, 2020 at 04:15 PM', 5, 'Located in the vibrant area of Los Angeles, CA 90023, this company truly stands out. "Great company. Amazing customer service and they always have what we need in stock. Sometimes, we’d ask to hold for future orders and they will! Miss Jane is very helpful and great communicator."', 'gmap_0')
('Faranak Rafizadeh', '2021-04-12 17:07:52', 5, 'Los Angeles is known for its vibrant culture and friendly atmosphere. "Nice people helpful."', 'gmap_0')
('Javier Perez', '2018-04-23 16:24:26', 5, 'I had a fantastic experience at this amazing spot in Los Angeles, CA 90023, where the friendly staff went above and beyond to make my visit truly enjoyable!', 'gmap_0')
('Luis P.', '2017-07-10 22:12:19', 5, 'I had an amazing experience at this charming café in Los Angeles, where the friendly staff and delicious pastries made my day truly special!', 'gmap_0')
('His Mama Cakez', 'May 19, 2021 at 03:5

In [3]:
import pandas as pd
import sqlite3

# 将 la_data 转为 DataFrame
la_df = pd.DataFrame(la_data)

# 提取所有 gmap_id
gmap_ids = la_df["gmap_id"].tolist()

# 连接 review_query.db 数据库
conn = sqlite3.connect("../query_dataset/review_query.db")

# 查询匹配 gmap_id 的所有评分记录
placeholder = ','.join('?' for _ in gmap_ids)
query = f"""
    SELECT gmap_id, rating
    FROM review
    WHERE gmap_id IN ({placeholder})
"""
review_df = pd.read_sql_query(query, conn, params=gmap_ids)

# 计算平均评分
avg_rating = review_df.groupby("gmap_id")["rating"].mean().reset_index()
avg_rating.rename(columns={"rating": "avg_rating"}, inplace=True)

# 合并回原始洛杉矶商家数据
la_with_rating = la_df.merge(avg_rating, on="gmap_id", how="left")

# 按平均评分降序排列，取前5名
top5 = la_with_rating.sort_values(by="avg_rating", ascending=False).head(5)

print(top5)


NameError: name 'la_data' is not defined

In [9]:
# mysql -u root -p -D ucb_db < ../query_dataset/business_description.sql
# mysql -u root -p -D ucb_db < ../query_googlelocal/query_dataset/business_description.sql
import mysql.connector
import openai
import time
import json
from tqdm import tqdm
from openai import AzureOpenAI

# ====== Azure OpenAI 配置 ======
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)

deployment_name = "gpt-4o-mini"  # 在 Azure Portal 的 Deployments 中查看


# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",
    database="ucb_db"
)
cursor = conn.cursor()

# 获取所有字段（包括 description）
cursor.execute("SELECT * FROM business_description")
column_names = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()

# GPT 判断是否是 massage therapist 的函数
def is_massage_therapist(description, client, deployment_name):
    prompt = (
        "Based on the following business description, determine whether it refers to a massage therapist.\n"
        "Only answer with 'yes' or 'no'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that identifies massage therapist businesses from descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower().startswith("yes")

    except Exception as e:
        print(f"❌ Error determining massage therapist from description: {e}")
        return False

# ✅ 用于保存 massage therapist 商家
massage_data = []

# ✅ 遍历记录进行 GPT 判断
for i, row in enumerate(rows[:100]):  # 可根据需要扩大范围
    record = dict(zip(column_names, row))
    description = record.get("description", "")

    if description and is_massage_therapist(description, client, deployment_name):
        massage_data.append(record)
        print(f"✅ [{i}] is Massage therapist: {description}")
    else:
        print(f"❌ [{i}] is not Massage therapist")

print(f"\n🎯 最终保留 {len(massage_data)} 条 Massage therapist 商家记录")


❌ [0] is not Massage therapist
❌ [1] is not Massage therapist
❌ [2] is not Massage therapist
❌ [3] is not Massage therapist
❌ [4] is not Massage therapist
❌ [5] is not Massage therapist
❌ [6] is not Massage therapist
✅ [7] is Massage therapist:  Experience relaxation and rejuvenation at this wellness retreat in Fair Oaks, CA 95628, where skilled therapists offer soothing treatments designed to relieve stress and promote overall well-being.
❌ [8] is not Massage therapist
✅ [9] is Massage therapist:  Located in Fair Oaks, CA 95628, this wellness studio offers expert bodywork services designed to promote relaxation and alleviate tension.
❌ [10] is not Massage therapist
✅ [11] is Massage therapist:  Experience rejuvenating bodywork and relaxation techniques at this wellness studio in Roseville, CA 95678.
✅ [12] is Massage therapist:  Offering a range of therapeutic bodywork services, this wellness center in Carmichael, CA 95608 helps clients relax and rejuvenate for overall well-being.
✅ [

In [10]:
import pandas as pd
import sqlite3

# 将 massage_data 转为 DataFrame
massage_df = pd.DataFrame(massage_data)

# 提取所有 gmap_id
gmap_ids = massage_df["gmap_id"].tolist()

# 连接 review_query.db 数据库
conn = sqlite3.connect("../query_dataset/review_query.db")

# 查询匹配 gmap_id 的所有评分记录
placeholder = ','.join('?' for _ in gmap_ids)
query = f"""
    SELECT gmap_id, rating
    FROM review
    WHERE gmap_id IN ({placeholder})
"""
review_df = pd.read_sql_query(query, conn, params=gmap_ids)

# 计算平均评分
avg_rating = review_df.groupby("gmap_id")["rating"].mean().reset_index()
avg_rating.rename(columns={"rating": "avg_rating"}, inplace=True)

# 合并平均评分回原始 massage 商家数据
massage_with_rating = massage_df.merge(avg_rating, on="gmap_id", how="left")

# 过滤出平均评分 ≥ 4.0 的商家
qualified_massage = massage_with_rating[ massage_with_rating["avg_rating"] >= 4.0 ].copy()

# 排序可选：按评分降序排列
qualified_massage = qualified_massage.sort_values(by="avg_rating", ascending=False)

# 输出最终结果
print(f"🎯 Total qualified Massage therapist businesses: {len(qualified_massage)}\n")
print(qualified_massage[["name", "gmap_id", "avg_rating", "description"]])  # 可视化字段可自定


🎯 Total qualified Massage therapist businesses: 4

               name  gmap_id  avg_rating  \
1     Elite Massage  gmap_25    5.000000   
0   Angel-A Massage  gmap_22    4.333333   
4    Aurora Massage  gmap_20    4.178571   
5  J B Oriental Inc  gmap_32    4.166667   

                                         description  
1   Located in Fair Oaks, CA 95628, this wellness...  
0   Experience relaxation and rejuvenation at thi...  
4   Located in Sacramento, CA 95821, this wellnes...  
5   Experience rejuvenating therapies and soothin...  


In [11]:
# mysql -u root -p -D ucb_db < ../query_dataset/business_description.sql
# mysql -u root -p -D ucb_db < ../query_googlelocal/query_dataset/business_description.sql
import mysql.connector
import openai
import time
import json
from tqdm import tqdm
from openai import AzureOpenAI

# ====== Azure OpenAI 配置 ======
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)

deployment_name = "gpt-4o-mini"  # 在 Azure Portal 的 Deployments 中查看


# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",
    database="ucb_db"
)
cursor = conn.cursor()

# 获取所有字段（包括 description）
cursor.execute("SELECT * FROM business_description")
column_names = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()

# GPT 判断是否是 massage therapist 的函数
def is_open_after_6pm_by_gpt(hours_string, client, deployment_name):
    prompt = (
        "Based on the weekly business hours provided below, determine whether this business is open after 6PM on any day.\n"
        "Only answer with 'yes' or 'no'.\n\n"
        f"Hours: {hours_string}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that reads weekly business hours and identifies late openings."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower().startswith("yes")
    
    except Exception as e:
        print(f"❌ Error checking hours with GPT: {e}")
        return False

# ✅ 保存营业到晚上 6PM 后的商家
open_late_data = []

for i, row in enumerate(rows[:100]):  # 你可以扩到更多
    record = dict(zip(column_names, row))
    hours = record.get("hours", "")

    if hours and is_open_after_6pm_by_gpt(hours, client, deployment_name):
        open_late_data.append(record)
        print(f"✅ [{i}] Open after 6PM: {hours}")
    else:
        print(f"❌ [{i}] Not open after 6PM")

print(f"\n🎯 Total businesses open after 6PM: {len(open_late_data)}")


❌ [0] Not open after 6PM
❌ [1] Not open after 6PM
❌ [2] Not open after 6PM
❌ [3] Not open after 6PM
❌ [4] Not open after 6PM
✅ [5] Open after 6PM: [["Thursday", "11AM–9:30PM"], ["Friday", "11AM–9:30PM"], ["Saturday", "11AM–9:30PM"], ["Sunday", "11AM–9:30PM"], ["Monday", "Closed"], ["Tuesday", "11AM–9:30PM"], ["Wednesday", "11AM–9:30PM"]]
❌ [6] Not open after 6PM
✅ [7] Open after 6PM: [["Thursday", "9:30AM–9:30PM"], ["Friday", "9:30AM–9:30PM"], ["Saturday", "9:30AM–9:30PM"], ["Sunday", "10AM–8PM"], ["Monday", "10AM–9:30PM"], ["Tuesday", "10AM–9:30PM"], ["Wednesday", "9:30AM–9:30PM"]]
❌ [8] Not open after 6PM
❌ [9] Not open after 6PM
❌ [10] Not open after 6PM
✅ [11] Open after 6PM: [["Thursday", "9:30AM–10PM"], ["Friday", "9:30AM–10PM"], ["Saturday", "9:30AM–10PM"], ["Sunday", "9:30AM–10PM"], ["Monday", "9:30AM–10PM"], ["Tuesday", "9:30AM–10PM"], ["Wednesday", "9:30AM–10PM"]]
✅ [12] Open after 6PM: [["Thursday", "9AM–10PM"], ["Friday", "9AM–10PM"], ["Saturday", "9AM–10PM"], ["Sunday", "9

In [13]:
import pandas as pd
import sqlite3

# Step 1: Convert GPT-identified late open businesses to DataFrame
open_late_df = pd.DataFrame(open_late_data)

# Step 2: Extract all gmap_ids
gmap_ids = open_late_df["gmap_id"].tolist()

# Step 3: Connect to review database and query ratings
conn = sqlite3.connect("../query_dataset/review_query.db")
placeholder = ','.join(['?'] * len(gmap_ids))

query = f"""
    SELECT gmap_id, rating
    FROM review
    WHERE gmap_id IN ({placeholder})
"""
review_df = pd.read_sql_query(query, conn, params=gmap_ids)

# Step 4: Compute average rating per gmap_id
avg_rating = review_df.groupby("gmap_id")["rating"].mean().reset_index()
avg_rating.rename(columns={"rating": "avg_rating"}, inplace=True)

# Step 5: Merge ratings back into business data
open_late_with_rating = open_late_df.merge(avg_rating, on="gmap_id", how="left")

# Step 6: Sort by average rating and get top 5
top5_open_late = open_late_with_rating.sort_values(by="avg_rating", ascending=False).head(10)

# Step 7: Display result
print("🏆 Top 5 businesses open after 6PM by average rating:\n")
print(top5_open_late[["name", "hours", "avg_rating"]])


🏆 Top 5 businesses open after 6PM by average rating:

                           name  \
12             Taba Rug Gallery   
13       Beauty Divine Artistry   
14         White Barn Candle Co   
16              TACOS LA CABANA   
17          Mariscos el poblano   
18              Paradise tattoo   
9   The Boochyard @ Local Roots   
23            Widows Peak Salon   
8                The Beauty Bar   
15         Rossy's Beauty Salon   

                                                hours  avg_rating  
12  [["Thursday", "10AM–7PM"], ["Friday", "10AM–7P...    5.000000  
13  [["Thursday", "9AM–8PM"], ["Friday", "9AM–8PM"...    5.000000  
14  [["Thursday", "10AM–9PM"], ["Friday", "10AM–9P...    5.000000  
16  [["Thursday", "Closed"], ["Friday", "5–11PM"],...    5.000000  
17  [["Thursday", "Open 24 hours"], ["Friday", "8A...    5.000000  
18  [["Thursday", "12–10PM"], ["Friday", "12PM–12A...    4.960317  
9   [["Thursday", "3–8PM"], ["Friday", "3–9PM"], [...    4.894737  
23  [["Thursday"